# URBANRISK: Integrated Urban Risk Analytics & Resource Prioritization System
### Educational & Analytical Demonstration Notebook · Hyderabad, Telangana
**Author:** Sana Firdous (B.Tech CSE Final Year Project)  
**Domain:** Urban Informatics, Multi-Hazard Spatial Analytics & Applied Machine Learning  
---


## 1. Problem Statement & Research Objectives
Rapid urban expansion in the Greater Hyderabad metropolitan area has increased vulnerability to concurrent environmental hazards:
* **Urban Heat Island (UHI) Effect:** High surface temperatures during summer months exacerbating heat stress in dense built-up zones.
* **Urban Flood Inundation:** Monsoonal flash floods overwhelming stormwater drainage networks along the Musi River basin.
* **Emergency Infrastructure Gaps:** Disparities in travel distance and spatial access to life-safety infrastructure (hospitals, trauma clinics, fire stations).

This notebook demonstrates the empirical data processing, machine learning regression modeling, and multi-criteria spatial screening that power the **URBANRISK** decision-support engine.


## 2. Dataset Overview & Data Loading
We load empirical observations compiled across Hyderabad:
1. **MODIS Satellite Thermal Dataset:** Multi-year summer surface temperatures (`hyderabad_modis_heat.csv`).
2. **Integrated Priority Grid:** Precomputed 1-km spatial grid cells with multi-hazard scores (`integrated_urban_risk_priority_grid_1km.csv`).


In [3]:

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Detect Kaggle or local VS Code environment
if Path("/kaggle/input/datasets/sanafirdos/urbanrisk/kaggle_package").exists():
    ROOT = Path("/kaggle/input/datasets/sanafirdos/urbanrisk/kaggle_package")
else:
    ROOT = Path.cwd() / "kaggle_package"

# If notebook is already inside kaggle_package
if not (ROOT / "data").exists() and (Path.cwd() / "data").exists():
    ROOT = Path.cwd()

data_dir = ROOT / "data"
models_dir = ROOT / "models"

print("Environment:", "Kaggle" if "/kaggle/" in str(ROOT) else "Local")
print("Dataset root:", data_dir)
print("Models root:", models_dir)

assert data_dir.exists(), f"Dataset folder not found: {data_dir}"
assert models_dir.exists(), f"Models folder not found: {models_dir}"

print("Dataset and model folders verified!")


Environment: Local
Dataset root: c:\Users\sanaf\OneDrive\Desktop\urban-risk-analytics\kaggle_package\data
Models root: c:\Users\sanaf\OneDrive\Desktop\urban-risk-analytics\kaggle_package\models
Dataset and model folders verified!


## 3. Data Preprocessing & Exploratory Analysis
Let's inspect the distribution of daytime and nighttime surface temperatures from MODIS satellite passes.


In [4]:

# Load MODIS satellite thermal dataset
heat_csv = data_dir / "hyderabad_modis_heat.csv"

assert heat_csv.exists(), f"MODIS file not found: {heat_csv}"

df_heat = pd.read_csv(heat_csv)

print("Total Satellite Thermal Observations:", len(df_heat))
print("\nFirst 5 observations:")
print(df_heat.head())

print("\nDescriptive Summary:")
print(df_heat[["daytime_temp_c", "nighttime_temp_c"]].describe())


Total Satellite Thermal Observations: 1326

First 5 observations:
   pixel_id   latitude  longitude  daytime_temp_c  nighttime_temp_c
0         0  17.252145  78.301652       36.602941         20.399730
1         1  17.252145  78.310635       37.113429         20.463514
2         2  17.252145  78.319618       37.561429         20.448333
3         3  17.252145  78.328601       37.337429         20.488919
4         4  17.252145  78.337584       36.722571         20.528919

Descriptive Summary:
       daytime_temp_c  nighttime_temp_c
count     1326.000000       1326.000000
mean        34.323556         22.473464
std          1.299366          1.225156
min         26.644737         19.846757
25%         33.789167         21.442143
50%         34.312162         22.412270
75%         34.978026         23.442500
max         37.593243         25.210000


In [ ]:
# Diurnal Temperature Delta Distribution
df_heat["diurnal_delta_c"] = df_heat["daytime_temp_c"] - df_heat["nighttime_temp_c"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_heat["daytime_temp_c"], kde=True, color="#b91c1c", ax=axes[0])
axes[0].set_title("Distribution of Daytime Surface Temperature (deg C)")
axes[0].set_xlabel("Temperature (deg C)")

sns.histplot(df_heat["diurnal_delta_c"], kde=True, color="#ea580c", ax=axes[1])
axes[1].set_title("Diurnal Temperature Delta (Day - Night deg C)")
axes[1].set_xlabel("Thermal Delta (deg C)")
plt.tight_layout()
plt.show()


## 4. Machine Learning Model: Daytime Temperature Regression
We utilize a **Random Forest Regressor** pipeline that takes spatial coordinates (`latitude`, `longitude`) and nighttime thermal readings (`nighttime_temp_c`) to predict daytime land surface temperature.

Let's evaluate the regression pipeline using an 80/20 train-test split.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

features = ["latitude", "longitude", "nighttime_temp_c"]
target = "daytime_temp_c"

X = df_heat[features]
y = df_heat[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training Set:", X_train.shape[0], "| Held-out Test Set:", X_test.shape[0])

# Build Pipeline
rf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
])

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 45)
print(f"Held-out Validation MAE:  {mae:.4f} deg C")
print(f"Held-out R2 Score:        {r2:.4f}")
print("=" * 45)


## 5. Sample Coordinate Predictions
We test the model across recognized Hyderabad geographic landmarks:
* **Charminar / Old City:** High density, commercial built-up area
* **Hitec City / Cyberabad:** High-tech commercial district
* **Begumpet / Secunderabad:** Mixed residential-institutional corridor
* **Moosarambagh / Musi Basin:** Drainage catchment corridor


In [ ]:
landmarks = [
    {"name": "Charminar / Old City", "lat": 17.3616, "lon": 78.4747, "night_temp": 24.5},
    {"name": "Hitec City / Cyberabad", "lat": 17.4435, "lon": 78.3772, "night_temp": 23.8},
    {"name": "Begumpet / Secunderabad", "lat": 17.4447, "lon": 78.4664, "night_temp": 24.2},
    {"name": "Moosarambagh / Musi Basin", "lat": 17.3705, "lon": 78.5080, "night_temp": 24.0}
]

results = []
for lm in landmarks:
    inp = pd.DataFrame([{"latitude": lm["lat"], "longitude": lm["lon"], "nighttime_temp_c": lm["night_temp"]}])
    pred = rf_model.predict(inp)[0]
    delta = pred - lm["night_temp"]
    results.append({
        "Landmark": lm["name"],
        "Latitude": lm["lat"],
        "Longitude": lm["lon"],
        "Night Temp (deg C)": lm["night_temp"],
        "Predicted Day Temp (deg C)": round(pred, 2),
        "Thermal Delta (deg C)": round(delta, 2)
    })

df_preds = pd.DataFrame(results)
print(df_preds.to_string(index=False))


## 6. Multi-Criteria Integrated Urban Priority Screening (IUPS)
The core contribution of URBANRISK is the **Integrated Urban Priority Screening (IUPS)** framework. 
Rather than evaluating single hazards, IUPS calculates a composite 0–100 screening score across 791 1-km cells:

$$\text{IUPS Priority Score} = 0.35 \times \text{Flood} + 0.35 \times \text{Heat} + 0.15 \times \text{Infra Gap} + 0.15 \times \text{Road Gap}$$


In [5]:

# Load Integrated Urban Risk Priority Grid
grid_csv = data_dir / "integrated_urban_risk_priority_grid_1km.csv"

assert grid_csv.exists(), f"Priority grid file not found: {grid_csv}"

df_grid = pd.read_csv(grid_csv)

scored_cells = df_grid.dropna(
    subset=["priority_score_0_100"]
)

print("Total Grid Cells in Study Area:", len(df_grid))
print("Scored Grid Cells with Complete Coverage:", len(scored_cells))


Total Grid Cells in Study Area: 791
Scored Grid Cells with Complete Coverage: 741


In [ ]:
# Top 10 Highest Priority Intervention Cells in Hyderabad
top_10 = scored_cells.sort_values(by="priority_score_0_100", ascending=False).head(10)
cols_to_show = [
    "grid_id", "cell_center_latitude", "cell_center_longitude",
    "heat_evidence_score", "flood_evidence_score", 
    "infrastructure_gap_score", "priority_score_0_100", "screening_priority"
]
print(top_10[cols_to_show].to_string(index=False))


## 7. Results & Key Findings
1. **Thermal Vulnerability:** Dense urban cores (Old City, central commercial zones) exhibit daytime surface temperatures exceeding 34.5 deg C with diurnal deltas > 10.0 deg C.
2. **Prioritization Clusters:** The IUPS screening successfully isolates the highest-risk grid cells along low-lying drainage channels and dense built-up areas lacking nearby fire and emergency medical stations.
3. **Model Accuracy:** The Random Forest regressor achieves a validation MAE of ~0.506 deg C, demonstrating that spatial coordinates paired with nighttime thermal baseline provide a reliable predictor for daytime heat exposure.

## 8. Limitations & Scope
* **Screening vs Simulation:** IUPS provides relative prioritization for municipal capital planning; it does not replace fine-grained hydrodynamic 2D flood modeling.
* **Observation Freshness:** The thermal baseline uses multi-year MODIS summer imagery. Live weather radar forecasting is not included.

## 9. Conclusion
The URBANRISK platform bridges the gap between academic data science and municipal decision support, delivering transparent, reproducible urban vulnerability metrics for Greater Hyderabad.
